# 🔍 The Case of the Corrupted Applications
### Data Assessing & Data Cleaning — learn it like a detective

> *"You see, but you do not observe."* — Sherlock Holmes
> *"It is a capital mistake to theorize before one has data."* — Sherlock Holmes

**The client:** a bank's credit team has been handed a loan-application extract (credit cards + personal loans). Someone suspects it's unreliable. **You are the detective.** You will (1) *assess* the data without touching it, (2) build an evidence board, (3) clean it with a disciplined method, and (4) prove — with tests — that the case is closed.

### Your teacher's framework (mapped to the casework)
```
DATA WRANGLING
 ├─ 1a Gathering   (last workbook: read_csv / read_excel / SQL / API …)
 ├─ 1b ASSESSING   ← Act I–III   "Observe, don't touch."   (the notes call this “Data Accessing”)
 └─ 1c CLEANING    ← Act IV–VI   "Define → Code → Test"
```
| Teacher's idea | Where you'll practise it |
|---|---|
| Dirty data (quality: duplicated, missing, corrupt, inaccurate) | Acts II–III |
| Messy data (tidiness: variable = column, observation = row, unit = table) | Acts II–III, Act V |
| Manual vs programmatic assessment | Act I |
| Quality dimensions: complete · valid · accurate · consistent | Act III evidence board |
| Cleaning order: Completeness → Tidiness → Validity → Accuracy → Consistency | Act IV–V |
| Define → Code → Test, always on a **copy** | every cleaning step |

> 📝 Terminology: in the wrangling literature this step is normally spelled **“assessing”** (judging quality). “Accessing” is *getting* the data (gathering). Your notes use it for assessing, so I'll use **Assessing** — just so you recognise both words in the wild.

### The detective's method (every single fix follows these 6 beats)
**Observe → Hypothesise → Test the hypothesis → Record it on the evidence board → Remedy → Verify.**

### How to learn from this notebook
🔮 **Predict** before you run · 🔧 **Break it** · ✍️ **Exercise** cells (fill `...`, then run the ✅ check) · 🧠 **Recall** questions *before* opening answers · 🗣️ **Explain it to Watson** (say it out loud in 3 sentences).

In [ ]:
import pandas as pd, numpy as np, re, difflib
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 30, "display.width", 140)
DATA = Path("../data")
print("pandas", pd.__version__, "|", sorted(p.name for p in DATA.iterdir()))

---
# ACT I · The crime scene — first impressions (Assessing, step 1)
Teacher's three-part start: **① write a summary of the data · ② write a description of each column · ③ add any additional information.**
Detectives do this *before* theorising.

### I.1 Look with ordinary eyes first (naive read)
**🔮 Predict:** which columns that *should* be numeric will pandas read as text? Write your guess: _..._

In [ ]:
df_naive = pd.read_csv(DATA / "loan_applications_raw.csv")      # the way most people load data
display(df_naive.head(3))
df_naive.info()

**Observe:** `annual_income` is text (`object`/`str`) — a numeric column that couldn't parse means *some cell isn't a number*. Your first clue.

### I.2 The forensic read — never let pandas “pre-interpret” the evidence
By default `read_csv` silently turns `""`, `"N/A"`, `"NA"`, `"null"` … into `NaN`. That **destroys information** (was it blank? or the text N/A?). A detective reads everything **as text, with nothing converted**, then decides.

In [ ]:
raw = pd.read_csv(DATA / "loan_applications_raw.csv", dtype=str, keep_default_na=False)   # ← the untouched evidence
print(raw.shape)
print("NaNs in naive read of bureau_score :", df_naive.bureau_score.isna().sum())
print("Literal 'N/A' strings in raw       :", (raw.bureau_score == "N/A").sum(), "  ← pandas had quietly converted these")

### I.3 Manual assessment — do this in Excel (10 minutes, don't skip)
Open `data/loan_applications_raw.xlsx`. Manual inspection catches things summaries hide. Your toolkit:
* **Filter dropdown** on a text column → lists every distinct value (typos and variants jump out).
* **Sort smallest→largest / largest→smallest** on numeric columns → impossible values collect at the ends.
* **Conditional formatting → Highlight duplicates** on `app_id`.
* **Freeze panes + scroll** the whole thing once, like reading a witness statement.

📝 **Write down 5 suspicious things you spotted manually** (we'll check them programmatically later):
1. _..._ 2. _..._ 3. _..._ 4. _..._ 5. _..._

### I.4 Programmatic assessment — the standard kit
`head()`, `sample()`, `info()`, `describe()`, `nunique()`, `value_counts()`, `isna()`, `duplicated()`. 
Use **`sample()`** as much as `head()` — the top rows are rarely representative.

In [ ]:
display(raw.sample(8, random_state=1).T)
print("\nUnique values per column:")
display(raw.nunique().to_frame("unique").T)

### I.5 Write your case summary (teacher's steps ①②③)
Official column descriptions are in the data dictionary. Read it — **then treat it as a set of claims to be tested**, not as truth.

In [ ]:
dictionary = pd.read_csv(DATA / "data_dictionary.csv")
display(dictionary.style.set_properties(**{"text-align": "left"}))

**① Summary** (what is this data? grain = one row per ___? rows? time span? source?): _..._

**② Column notes** (add anything the dictionary doesn't say, e.g. “`term` blank for credit cards — expected or missing?”): _..._

**③ Additional information** (what would you ask the data owner? e.g. “how is `default_flag` defined?”): _..._

> 🗣️ **Explain to Watson:** why do we read the data as text first? (Answer in your own words before moving on.)

---
# ACT II · Know your suspects — the two kinds of unclean data
| | **Dirty data** (content problems = *quality*) | **Messy data** (structure problems = *tidiness*) |
|---|---|---|
| Also called | low-quality data | untidy data |
| Examples | duplicated · missing · corrupt · inaccurate | a column holding two variables · column headers that are really values · two tables mashed into one |
| Rule that's broken | “the values should be right” | **each variable → a column · each observation → a row · each observational unit → a table** |

**Quality dimensions** (teacher's order of severity: *Completeness ← Validity ← Accuracy ← Consistency*):
| Dimension | Question it asks | Credit-risk example |
|---|---|---|
| **Complete** | Is anything missing? | income is blank |
| **Valid** | Does it obey the rules of the format/range? | bureau score 9999 (scale is 300–900) |
| **Accurate** | Is it *true*? (needs another source or another column to check) | stated age ≠ age from date of birth |
| **Consistent** | Is the same thing written the same way? | `NSW`, `nsw`, `New South Wales` |

> 🧠 **Key distinction:** *valid* ≠ *accurate*. “Age = 47” is valid (a possible age) but could still be wrong. Validity checks the **rule**; accuracy checks the **truth**.

### ✍️ Exercise 1 — Suspect line-up
Classify each symptom. For each, give `("dirty" or "messy", dimension)`. For messy issues, use `"tidiness"` as the dimension. Dimensions: `"complete" · "valid" · "accurate" · "consistent" · "unique" · "tidiness"`.

In [ ]:
lineup = {
    "income cell contains the text 'unknown'":                                  ("dirty", "complete"),
    "bureau_score = 9999 when the scale is 300–900":                            ("dirty", "valid"),
    "stated age is 52 but date of birth says the person is 31":                 ("dirty", "accurate"),
    "'NSW', 'nsw' and 'New South Wales' in the same column":                    ("dirty", "consistent"),
    "the same application appears twice":                                       ("dirty", "unique"),
    "columns dpd_m1 … dpd_m6 — the month number is stored in the header":       ("messy", "tidiness"),
    "approved_amount is larger than requested_amount":                          ("dirty", "accurate"),
    "one cell holds 'Olivia Nguyen | 1985-03-12' (name AND date of birth)":     ("messy", "tidiness"),
}

In [ ]:
expected = list(lineup.values())
truth = [("dirty","complete"),("dirty","valid"),("dirty","accurate"),("dirty","consistent"),("dirty","unique"),("messy","tidiness"),("dirty","accurate"),("messy","tidiness")]
wrong = [k for (k, v), t in zip(lineup.items(), truth) if v != t]
assert not wrong, f"Re-think these {len(wrong)} suspect(s): {wrong}"
print("✅ Exercise 1 passed")

*(Debatable one: “approved > requested” could also be argued as a **validity** rule from the dictionary. The point is not the label but being able to **justify** it: it's a cross-field truth check → accurate.)*

---
# ACT III · Interrogation — systematic assessment, dimension by dimension
We assess in the **same order we'll clean**, and every finding goes onto an **evidence board** — a DataFrame. Assessing is **iterative**: each clue can reveal new questions.

### The evidence board

In [ ]:
board = []
def log_issue(kind, dimension, column, description, n_rows):
    """Add a finding to the evidence board. kind: 'dirty'|'messy'; dimension: complete|valid|accurate|consistent|unique|tidiness"""
    board.append(dict(kind=kind, dimension=dimension, column=column, description=description, n_rows=n_rows))
    return pd.DataFrame(board).tail(1)

print("Evidence board ready. Log every finding with log_issue(...)")

## III.1 Duplicates (uniqueness)
**🔮 Predict:** `raw.duplicated().sum()` tells you how many *exact* duplicate rows exist. Will it find *all* repeated applications? Why / why not?

In [ ]:
n_exact = raw.duplicated().sum()
ids_normalised = raw.app_id.str.strip().str.upper()
print("exact duplicate rows            :", n_exact)
print("repeated application IDs (raw)  :", raw.app_id.duplicated().sum())
print("repeated IDs after strip+upper  :", ids_normalised.duplicated().sum())
display(raw[ids_normalised.isin(ids_normalised[ids_normalised.duplicated()])].assign(_id=ids_normalised).sort_values("_id").head(6)[["app_id", "product", "state", "annual_income", "_id"]])

**Deduction:** `duplicated()` compares *exact text*. A trailing space or different case makes two rows look different — **near-duplicates**. Always compare on a *normalised key* (here: stripped, upper-cased `app_id`).

In [ ]:
n_near_extra = ids_normalised.duplicated().sum() - raw.duplicated().sum()

In [ ]:
assert n_near_extra == 25, f"got {n_near_extra} — total repeated IDs after normalising minus exact duplicates"
print("✅ Exercise 2 passed")
log_issue("dirty", "unique", "app_id", "exact duplicate rows", 40)
log_issue("dirty", "unique", "app_id", "near-duplicates (whitespace/case in app_id)", n_near_extra)

## III.2 Completeness — *what is missing?*  (and what is **pretending** not to be?)
Missing values wear disguises: `""`, `N/A`, `unknown`, `-`, `0`, `-1`, `9999`. First list the usual suspects across all columns:

In [ ]:
MISSING_TOKENS = ["", "n/a", "na", "null", "none", "unknown", "-", "?"]
tok = raw.apply(lambda s: s.str.strip().str.lower().isin(MISSING_TOKENS)).sum()
display(tok[tok > 0].to_frame("cells that look missing"))
print("\nWhich tokens hide in annual_income?")
display(raw.annual_income.str.strip().str.lower().loc[lambda s: s.isin(MISSING_TOKENS)].value_counts().to_frame())

**🔮 Predict:** income also has *numeric* placeholders. Which values would a lazy exporter use for “no income”? Check the smallest numbers:

In [ ]:
inc_num = pd.to_numeric(raw.annual_income, errors="coerce")
print(inc_num.sort_values().head(3).tolist())
print(inc_num.value_counts().head(3).to_dict(), "  ← unusually frequent values are suspicious")

### ✍️ Exercise 3 — How many income cells are *really* missing?
Count rows where `annual_income` is a missing token (blank / N/A / unknown …, after strip+lower) **or** a numeric placeholder (a number ≤ 0).
Name it `n_income_missing_any`.

In [ ]:
is_token = raw.annual_income.str.strip().str.lower().isin(MISSING_TOKENS)
is_placeholder = pd.to_numeric(raw.annual_income, errors="coerce") <= 0
n_income_missing_any = int((is_token | is_placeholder).sum())

In [ ]:
assert n_income_missing_any == 112, f"got {n_income_missing_any} — did you include BOTH tokens and numbers <= 0?"
print("✅ Exercise 3 passed")
log_issue("dirty", "complete", "annual_income", "blank / N/A / unknown / 0 / -1 used for 'not provided'", n_income_missing_any)

### Not every blank is a problem — *structural* missingness
`term` is blank for many rows. Before panicking, **test the hypothesis**: “blank term ⇔ credit card”.

In [ ]:
prod_norm = raw["product"].str.strip().str.lower()
is_card = prod_norm.isin(["credit card", "cc"])
print(pd.crosstab(is_card.rename("is_credit_card"), raw.term.eq("").rename("term_blank")))

**Deduction:** all blanks are credit cards → term is *not applicable* there. Blank ≠ missing. **Never impute a structurally-missing value.** (Lesson: crosstab a missing-flag against other columns to find *why* it's missing.)

## III.3 Validity — *does it obey the rules?*  (rules come from the data dictionary)
### Numbers: the sentinel hunt

In [ ]:
score_num = pd.to_numeric(raw.bureau_score, errors="coerce")        # "N/A" → NaN here (just for inspecting)
print(score_num.describe().round(1).to_dict())
print("\nValues outside the documented 300–900 range:")
display(score_num[~score_num.between(300, 900) & score_num.notna()].value_counts().to_frame("rows"))

In [ ]:
n_invalid_scores = int((score_num.notna() & ~score_num.between(300, 900)).sum())

In [ ]:
assert n_invalid_scores == 43, f"got {n_invalid_scores}"
print("✅ Exercise 4 passed")
log_issue("dirty", "valid", "bureau_score", "9999 sentinel, 0 and other out-of-range values", n_invalid_scores)

### Dates: mixed formats, impossible dates, impossible years
**🔮 Predict:** the column mixes 3 text formats. If we parse each format separately and combine, what will remain `NaT` (not-a-time)?

In [ ]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d %b %Y"]
def parse_dates_multi(s):
    """Try each known format in turn; anything that fits none of them stays NaT."""
    out = pd.to_datetime(s, format=DATE_FORMATS[0], errors="coerce")
    for f in DATE_FORMATS[1:]:
        out = out.fillna(pd.to_datetime(s, format=f, errors="coerce"))
    return out

dates = parse_dates_multi(raw.application_date)
print("unparseable (impossible calendar dates):", dates.isna().sum())
print(raw.application_date[dates.isna()].head(6).tolist())
print("parsed range:", dates.min().date(), "→", dates.max().date(), "  ← dictionary says 2023-01-01 … 2025-06-30")

In [ ]:
n_bad_dates = int((dates.isna() | ~dates.between("2023-01-01", "2025-06-30")).sum())

In [ ]:
assert n_bad_dates == 20, f"got {n_bad_dates} — remember NaT must count too (NaT.between(...) is False, so use ~between)"
print("✅ Exercise 5 passed")
log_issue("dirty", "valid", "application_date", "impossible calendar dates + years outside 2023–2025", n_bad_dates)

### Text that should be a number (corrupt format)

In [ ]:
not_num = inc_num.isna() & ~raw.annual_income.str.strip().str.lower().isin(MISSING_TOKENS)
print("income cells that are neither a number nor a missing-token:", not_num.sum())
print(raw.annual_income[not_num].sample(6, random_state=2).tolist())
log_issue("dirty", "valid", "annual_income", "numbers stored as text: '85k' and '$85,000'", int(not_num.sum()))

## III.4 Accuracy — *is it true?*  (use a second source: another column, or a rule)
Accuracy can't be seen in one column. We **cross-examine** columns against each other.

**Alibi 1 — age vs date of birth.** DOB is hidden inside `applicant_info`. Let's extract it for now (lightly) and recompute age on the application date.

In [ ]:
dob = pd.to_datetime(raw.applicant_info.str.split("|").str[1].str.strip(), format="%Y-%m-%d")

def age_on(date, dob):
    """Whole years between dob and date."""
    return (date.dt.year - dob.dt.year) - (((date.dt.month < dob.dt.month) | ((date.dt.month == dob.dt.month) & (date.dt.day < dob.dt.day))).astype(int))

in_window = dates.between("2023-01-01", "2025-06-30")           # an age can't be trusted if the date itself is invalid (dependency!)
age_calc = age_on(dates.where(in_window), dob)
age_mismatch = (age_calc != pd.to_numeric(raw.age)) & age_calc.notna()
print("age ≠ age implied by DOB:", age_mismatch.sum())
display(pd.DataFrame({"stated": raw.age, "from_dob": age_calc})[age_mismatch].head(4))

In [ ]:
assert int(age_mismatch.sum()) == 30
log_issue("dirty", "accurate", "age", "stated age disagrees with DOB + application date", int(age_mismatch.sum()))

**Alibi 2 — approved vs requested.** An approved amount above the request is suspicious.
**Alibi 3 — `default_flag` vs the DPD columns.** The dictionary *defines* default as “any DPD ≥ 90 in months 1–6”. So the flag is *derivable* — a perfect cross-check.

### ✍️ Exercise 6 — Two cross-examinations
Compute: `n_approved_gt_requested` and `n_default_conflict` (rows where the stated flag disagrees with the DPD-derived flag).
*Hints:* convert text to numbers first. The flag mixes `0/1` with `Y/N` — map them to ints (`{"Y":1,"N":0,"1":1,"0":0}`).

In [ ]:
dpd_cols = [f"dpd_m{k}" for k in range(1, 7)]
n_approved_gt_requested = int((pd.to_numeric(raw.approved_amount) > pd.to_numeric(raw.requested_amount)).sum())
derived = (raw[dpd_cols].apply(pd.to_numeric).max(axis=1) >= 90).astype(int)
stated = raw.default_flag.map({"Y": 1, "N": 0, "1": 1, "0": 0})
n_default_conflict = int((derived != stated).sum())

In [ ]:
assert n_approved_gt_requested == 35, f"approved>requested: got {n_approved_gt_requested}"
assert n_default_conflict == 25, f"default conflicts: got {n_default_conflict}"
print("✅ Exercise 6 passed")
log_issue("dirty", "accurate", "approved_amount", "approved amount exceeds requested amount", n_approved_gt_requested)
log_issue("dirty", "accurate", "default_flag", "stated flag contradicts DPD history", n_default_conflict)

**Alibi 4 — income scale.** Plot-free trick: look at the *shape* of the distribution of numeric incomes.

In [ ]:
pos = inc_num[inc_num > 0]
print(pos.describe().round(0).to_dict())
print("\nincomes between 1 and 1,000 (nobody earns $85 a year):", int(pos.between(0.001, 1000).sum()))
print(pos[pos < 1000].head(5).tolist(), "  ← looks like income *in thousands*")
log_issue("dirty", "accurate", "annual_income", "some values appear to be in $ thousands (e.g. 85.3)", int(pos.between(0.001, 1000).sum()))

## III.5 Consistency — *same thing, same spelling?*
**Categorical columns: let the filter-dropdown logic run in code.**

In [ ]:
for col in ["state", "product", "employment_status"]:
    vc = raw[col].value_counts()
    print(f"── {col}: {len(vc)} distinct spellings for what should be a short list")
    print(vc.head(14).to_dict(), "\n")

**🔮 Predict:** how many *distinct spellings* does `state` have for the 8 real states? After you guess, run `raw.state.nunique()`.

In [ ]:
print("state spellings:", raw.state.nunique(), "| product:", raw["product"].nunique(), "| employment:", raw.employment_status.str.strip().nunique())
log_issue("dirty", "consistent", "state", "abbreviations, lowercase, full names, UPPERCASE for the same 8 states", raw.state.nunique())
log_issue("dirty", "consistent", "product", "'CC', 'credit card', 'P. Loan', 'PL' … for 2 products", raw["product"].nunique())
log_issue("dirty", "consistent", "employment_status", "case + padding variants", raw.employment_status.nunique())
log_issue("dirty", "valid", "employment_status", "corrupted text (letters swapped for digits, e.g. 'Fu11-time')", 25)
log_issue("dirty", "consistent", "application_date", "3 different date formats in one column", 3)
log_issue("dirty", "consistent", "default_flag", "mixes 0/1 with Y/N", int(raw.default_flag.isin(["Y", "N"]).sum()))
log_issue("dirty", "consistent", "term", "'36 months', '36 mths', '3 yrs' — unit and spelling vary", 3)

## III.6 Tidiness — *the three rules*
| Rule | Test | Verdict here |
|---|---|---|
| Each variable forms a column | any column holding 2 things? | `applicant_info` = name + DOB ❌ |
| Each observation forms a row | any column *headers* that are values? | `dpd_m1…dpd_m6`: month number is a value in the header ❌ |
| Each observational unit forms a table | do attributes of one entity repeat on many rows? | customer attributes (name, DOB, state) repeat for customers with several applications ❌ |

In [ ]:
per_cust = raw.assign(cid=raw.customer_id.str.strip()).drop_duplicates("app_id").groupby("cid").size()
print("customers with >1 application:", int((per_cust > 1).sum()), "→ their name/DOB/state are copied onto every application row")
log_issue("messy", "tidiness", "applicant_info", "two variables (name, DOB) in one column", len(raw))
log_issue("messy", "tidiness", "dpd_m1..dpd_m6", "variable ('month on book') stored in column headers", 6)
log_issue("messy", "tidiness", "customer attrs", "customer and application are two observational units in one table", int((per_cust > 1).sum()))

### ✍️ Exercise 7 — Finish the evidence board
Look at `pd.DataFrame(board)` below. Your teacher's rule: a good assessment documents **every dimension**. Anything missing? Add at least one issue you found **manually in Excel** (section I.3) that isn't on the board using `log_issue(...)`.

In [ ]:
evidence = pd.DataFrame(board)
display(evidence)
print(evidence.groupby(["kind", "dimension"]).size())

In [ ]:
log_issue("dirty", "complete", "term", "blank for credit cards — structural (NOT a defect; do not impute)", int(raw.term.eq("").sum()))

In [ ]:
evidence = pd.DataFrame(board)
assert len(evidence) >= 20, f"only {len(evidence)} entries — did you re-run the Act III cells top to bottom?"
assert {"complete", "valid", "accurate", "consistent", "unique", "tidiness"} <= set(evidence.dimension), "some dimensions missing from the board"
print(f"✅ Evidence board has {len(evidence)} findings across all dimensions")

### 🧠 Recall (Act I–III) — answer from memory
1. Why read as text with `keep_default_na=False` first?  2. Difference between *valid* and *accurate*?  3. How do you find near-duplicates?  4. Is a blank always “missing”?  5. Which tidy-data rule does `dpd_m1…dpd_m6` break?

<details><summary>Answers</summary>

1. Pandas silently converts `""`/`N/A`/… into `NaN` and guesses types — it destroys evidence. Read raw, then decide.
2. Valid = obeys the rules (format/range). Accurate = actually true (needs a second source or cross-check).
3. Normalise a key (`strip().upper()`), then `duplicated()` on the normalised key.
4. No — *structurally* missing (not applicable) differs from missing-at-random or hidden placeholders. Find out *why* (crosstab).
5. “Each variable forms a column”: the variable *month* lives in the headers → melt to long format.
</details>

---
# ACT IV · The plan — what order, and why?
Turn the evidence board into a **cleaning plan** using the teacher's order:

| # | Phase | Why this order (teacher's logic) |
|---|---|---|
| 0 | **Copy the data** | `raw` stays untouched as evidence; if a fix goes wrong we replay |
| 1 | Quality → **Completeness** (+ remove duplicates) | can't judge correctness of values that aren't there; duplicates would double-count everything |
| 2 | **Tidiness** | restructure *once* so later fixes apply to the right table/column |
| 3 | Quality → **Validity** | make every value legal (types, ranges, formats) |
| 4 | Quality → **Accuracy** | needs valid values to cross-check (can't compare an age to a DOB that is text) |
| 5 | Quality → **Consistency** | finally, one spelling per concept |

Each fix uses **Define → Code → Test**:
* **Define** – one sentence: *what* is wrong, *what* the fix is, *what assumption* we make.
* **Code** – the smallest code that does it.
* **Test** – an `assert` that **proves** it worked (never trust the code you just wrote).

In [ ]:
plan = pd.DataFrame(board)
order = {"complete": 1, "tidiness": 2, "unique": 1, "valid": 3, "accurate": 4, "consistent": 5}
plan["phase"] = plan.dimension.map(order)
display(plan.sort_values(["phase", "column"])[["phase", "dimension", "column", "description", "n_rows"]].reset_index(drop=True))

> 🕵️ **Detective's caveat — the order is a guide, not a law.** Real data has *dependencies* between fixes. We'll hit three of them; each is a lesson:
> 1. Near-duplicates are only visible after **normalising IDs** (a light consistency fix) → we do a *minimal* normalisation inside step 1.
> 2. Fixing **validity** (e.g. 9999 → NaN) *creates new missing values* → we must **re-check completeness** afterwards. Assessing is iterative.
> 3. To **impute** missing income sensibly we need clean employment groups (a consistency fix) → imputation waits until the end ("completeness, round 2"). And de-duplicating the *customer table* needs standardised `state` first.

### Step 0 — Work on a copy

In [ ]:
work = raw.copy()           # `raw` = evidence locker, `work` = operating table
assert work is not raw and work.equals(raw)
print("rows:", len(work))

---
# ACT V · The remedy — Define → Code → Test
## Phase 1 — Completeness (and duplicates)
### 1a · Duplicates *(✍️ you do it — Exercise 8)*
**Define:** `65` extra rows are repeats of the same application (exact + whitespace/case variants of `app_id`). **Fix:** normalise `app_id`, keep the first row per ID. **Assumption:** repeats carry no new information (checked: they're copies).

In [ ]:
work["app_id"] = work.app_id.str.strip().str.upper()
work = work.drop_duplicates(subset="app_id", keep="first").reset_index(drop=True)

In [ ]:
assert isinstance(work, pd.DataFrame), "work must stay a DataFrame: assign the de-duplicated frame back to `work` (then re-run every cell below)"
assert len(work) == 1400, f"expected 1400 unique applications, got {len(work)}"
assert work.app_id.is_unique and work.app_id.str.fullmatch(r"APP-\d{4}").all()
print("✅ Exercise 8 passed — duplicates removed")

### 1b · Disguised missing values → real `NaN`  *(worked example — study it, then predict-and-modify)*
**Define:** blanks, `N/A`, `unknown`, `-`, `?` are *missing*; numeric placeholders `0` / `-1` in income also mean “not provided”. **Fix:** convert all to `NaN` and keep a **flag column** so the information “this was missing” survives. **Assumption:** `0` income never means a genuinely zero-income applicant (the dictionary says income must be > 0).

In [ ]:
def tokens_to_nan(s):
    return s.where(~s.str.strip().str.lower().isin(MISSING_TOKENS), np.nan)

work = work.apply(tokens_to_nan)                                 # all columns are still text
inc_try = pd.to_numeric(work.annual_income, errors="coerce")
work.loc[inc_try <= 0, "annual_income"] = np.nan                  # numeric placeholders (0, -1)
work["income_missing"] = work.annual_income.isna()
work["employment_missing"] = work.employment_status.isna()
work["score_na"] = work.bureau_score.isna()
print("income missing:", work.income_missing.sum(), "| employment missing:", work.employment_missing.sum(), "| score 'N/A':", work.score_na.sum())

In [ ]:
assert work.income_missing.sum() == 112, "income missing count differs from the assessed number"
print("✅ Test passed: every disguised-missing income now NaN and flagged")

> ⚠️ We deliberately **did not impute yet** (see caveat 3). *Note:* `term` blanks for credit cards were also converted to `NaN` — that's correct: not-applicable *is* NaN; we'll test it as structural later.

## Phase 2 — Tidiness
### 2a · One column, two variables → split `applicant_info` *(worked example)*
**Define:** `"name | YYYY-MM-DD"` holds two variables. **Fix:** split on `|`, strip, parse DOB. **Test:** both new columns non-null, no `|` left.

In [ ]:
parts = work.applicant_info.str.split("|", expand=True)
work["customer_name"] = parts[0].str.strip()
work["dob"] = pd.to_datetime(parts[1].str.strip(), format="%Y-%m-%d")
work = work.drop(columns="applicant_info")

assert work.customer_name.notna().all() and work.dob.notna().all()
assert not work.customer_name.str.contains(r"\|").any()
print("✅ split done:", work[["customer_id", "customer_name", "dob"]].head(2).values.tolist())

### 2b · Values in column headers → **melt** `dpd_m1…dpd_m6`  *(✍️ Exercise 9)*
**Define:** the month on book is a *variable* stored in headers. **Fix:** build a long table `performance` with columns `app_id, month_on_book (int 1–6), dpd (int)` — one row per application-month. Keep `work` intact for now (we need `max dpd` later).
*Hint:* `pd.melt(..., id_vars=, value_vars=, var_name=, value_name=)`, then `str.extract(r"(\d+)")` for the month number.

In [ ]:
dpd_cols = [f"dpd_m{k}" for k in range(1, 7)]
performance = work.melt(id_vars="app_id", value_vars=dpd_cols, var_name="month_on_book", value_name="dpd")
performance["month_on_book"] = performance.month_on_book.str.extract(r"(\d+)", expand=False).astype(int)
performance["dpd"] = performance.dpd.astype(int)

In [ ]:
assert list(performance.columns) == ["app_id", "month_on_book", "dpd"], f"columns: {list(performance.columns)}"
assert len(performance) == 8400, f"expected 8400 rows (1400 apps × 6 months), got {len(performance)}"
assert performance.month_on_book.between(1, 6).all() and performance.dpd.dtype.kind == "i"
print("✅ Exercise 9 passed — tidy performance table")

### 2c · Two observational units in one table → separate `customers`  *(do it after state is standardised — see caveat 3)*
We'll create it in Phase 5, once `state` has one spelling. 📌 *Remember this pending task.*

## Phase 3 — Validity
### 3a · Dates *(worked example)*
**Define:** 3 formats + impossible dates + out-of-window years. **Fix:** parse with all 3 formats; set anything unparseable or outside 2023-01-01…2025-06-30 to `NaT`; keep a flag. **Assumption:** we can't *recover* the true date → we mark it unknown instead of guessing.

In [ ]:
d = parse_dates_multi(work.application_date)
valid_date = d.between("2023-01-01", "2025-06-30")
work["application_date"] = d.where(valid_date)
work["date_invalid"] = ~valid_date

assert work.date_invalid.sum() == int(d.isna().sum() + (d.notna() & ~valid_date).sum())
assert str(work.application_date.dtype).startswith("datetime")
print("invalid dates set to NaT:", work.date_invalid.sum())

In [ ]:
assert work.date_invalid.sum() == 20, work.date_invalid.sum()
print("✅ Test passed")

### 3b · Money stored as text → numbers  *(✍️ Exercise 10)*
**Define:** `annual_income` contains `"85k"`, `"$85,000"`, `"73300"`, `"85.3"`, or `NaN`. **Fix:** write `parse_money(x)` returning a float: strip `$` and `,`; a trailing `k` means ×1000; `NaN` stays `NaN`. Apply to `annual_income`, `requested_amount`, `approved_amount`.
**Test:** dtype float; no text left. *(Don't fix the `85.3` "in thousands" values here — that's an **accuracy** fix, Phase 4.)*

In [ ]:
def parse_money(x):
    if pd.isna(x):
        return np.nan
    t = str(x).strip().lower().replace("$", "").replace(",", "")
    mult = 1000 if t.endswith("k") else 1
    return float(t.rstrip("k")) * mult

for c in ["annual_income", "requested_amount", "approved_amount"]:
    work[c] = work[c].map(parse_money).astype(float)

In [ ]:
assert all(work[c].dtype == "float64" for c in ["annual_income", "requested_amount", "approved_amount"]), "all three must be float64"
assert work.annual_income.isna().sum() == 112, "NaN count must be unchanged by parsing"
assert parse_money("$85,000") == 85000 and parse_money("85k") == 85000 and np.isnan(parse_money(np.nan))
print("✅ Exercise 10 passed")

### 3c · Out-of-range bureau scores *(✍️ Exercise 11)*
**Define:** scale is 300–900; `9999` is a **sentinel** (system code for “no score”), `0` and 150/250/1000/1200 are impossible. **Fix:** convert to numeric; anything outside 300–900 → `NaN`; keep flag `score_invalid`. Make `bureau_score` a nullable integer (`"Int64"`).
**⚠️ Dependency:** this *creates* new missing values — re-check completeness afterwards.

In [ ]:
score = pd.to_numeric(work.bureau_score, errors="coerce")
work["score_invalid"] = score.notna() & ~score.between(300, 900)
work["bureau_score"] = score.where(score.between(300, 900)).astype("Int64")

In [ ]:
assert work.score_invalid.sum() == 43, work.score_invalid.sum()
assert work.bureau_score.dropna().between(300, 900).all()
assert work.bureau_score.isna().sum() == 85, "total missing scores should be N/A + invalid"
print("✅ Exercise 11 passed — and note: missing scores rose from 42 to 85. That's the completeness re-check!")

### 3d · `term` as text → integer months *(✍️ Exercise 12)*
**Define:** `"36 months"`, `"36 mths"`, `"3 yrs"`, or `NaN` (cards). **Fix:** new numeric column `term_months`: number from the text; if unit is years → ×12. Drop `term`.

In [ ]:
def parse_term(x):
    if pd.isna(x):
        return np.nan
    n, unit = re.match(r"\s*(\d+)\s*([a-z]+)", str(x).lower()).groups()
    return float(n) * (12 if unit.startswith("y") else 1)

work["term_months"] = work.term.map(parse_term)
work = work.drop(columns="term")

In [ ]:
assert set(work.term_months.dropna().unique()) <= {12, 24, 36, 48, 60}, work.term_months.dropna().unique()
assert work.term_months.notna().sum() > 400
print("✅ Exercise 12 passed")

## Phase 4 — Accuracy
### 4a · Age — recompute from DOB *(worked example)*
**Define:** stated `age` disagrees with DOB for some rows. **Fix:** replace with the age implied by DOB + application date. **Assumption (document it!):** DOB is more trustworthy than a typed age (it comes from ID documents). If the application date is invalid we can't compute → `NaN`.

In [ ]:
stated_age = pd.to_numeric(work.age)
age_calc = age_on(work.application_date, work.dob)
work["age_corrected"] = (stated_age != age_calc) & age_calc.notna()
work["age"] = age_calc.astype("Int64")

assert work.age_corrected.sum() == int(((stated_age != age_calc) & age_calc.notna()).sum())
assert work.age.dropna().between(18, 100).all()
print("ages corrected:", work.age_corrected.sum(), "| age unknown (bad date):", work.age.isna().sum())

In [ ]:
assert work.age_corrected.sum() == 30
assert work.age.isna().sum() == 20
print("✅ Test passed")

### 4b · Approved > requested → **quarantine**, don't invent  *(✍️ Exercise 13)*
**Define:** approved amount exceeds the request. We **can't know** which number is wrong. **Fix:** (1) copy offending rows to a `review_queue` table for the business to resolve, (2) set their `approved_amount` to `NaN`, (3) add flag `approved_suspect`. **Principle:** *when truth can't be determined, quarantine and flag — never fabricate.*

In [ ]:
mask = work.approved_amount > work.requested_amount
review_queue = work.loc[mask, ["app_id", "requested_amount", "approved_amount"]].copy()
work["approved_suspect"] = mask
work.loc[mask, "approved_amount"] = np.nan

In [ ]:
assert len(review_queue) == 35, len(review_queue)
assert not (work.approved_amount > work.requested_amount).any(), "violations remain"
assert work.approved_suspect.sum() == 35
print("✅ Exercise 13 passed")

### 4c · Default flag — derive it from its definition  *(✍️ Exercise 14)*
**Define:** the dictionary says default = any DPD ≥ 90 in months 1–6. The stored flag contradicts that in some rows (and mixes `Y/N`). **Fix:** compute `default_flag` from `performance` (max dpd per app ≥ 90) — the raw flag becomes a *witness*, not the truth. Add `default_flag_corrected` where the old flag (after mapping `Y/N/1/0` to ints) disagreed.

In [ ]:
max_dpd = performance.groupby("app_id").dpd.max()
old_flag = work.default_flag.map({"Y": 1, "N": 0, "1": 1, "0": 0})
work["default_flag"] = (work.app_id.map(max_dpd) >= 90).astype(int)
work["default_flag_corrected"] = work.default_flag != old_flag

In [ ]:
assert work.default_flag.isin([0, 1]).all()
assert work.default_flag_corrected.sum() == 25, work.default_flag_corrected.sum()
print("✅ Exercise 14 passed — default rate now:", f"{work.default_flag.mean():.1%}")

### 4d · Income in thousands  *(✍️ Exercise 15)*
**Define:** values like `85.3` are annual income in **$ thousands**. **Fix:** any income between 0 and 1,000 → ×1,000; add flag `income_rescaled`. **Assumption:** nobody legitimately earns < $1,000/year (dictionary range is 10,000–1,000,000).

In [ ]:
small = work.annual_income.between(0, 1000, inclusive="neither")
work["income_rescaled"] = small
work.loc[small, "annual_income"] = work.loc[small, "annual_income"] * 1000

In [ ]:
assert work.income_rescaled.sum() == 30, work.income_rescaled.sum()
assert work.annual_income.dropna().min() >= 10_000, work.annual_income.min()
print("✅ Exercise 15 passed")

## Phase 5 — Consistency
### 5a · Categories: product & state  *(✍️ Exercise 16)*
**Define:** many spellings per concept. **Fix:** *mapping dictionaries* applied to a **normalised key** (`strip().lower()`). **Test:** the result values ⊆ the allowed list — and **unmapped values must raise an alarm**, not pass silently (`map` returns NaN for unknown keys → assert none appear).

In [ ]:
STATES = {"NSW": "New South Wales", "VIC": "Victoria", "QLD": "Queensland", "WA": "Western Australia",
          "SA": "South Australia", "TAS": "Tasmania", "ACT": "Australian Capital Territory", "NT": "Northern Territory"}

product_map = {"credit card": "Credit Card", "cc": "Credit Card", "personal loan": "Personal Loan", "p. loan": "Personal Loan", "pl": "Personal Loan"}
state_map = {**{a.lower(): a for a in STATES}, **{full.lower(): a for a, full in STATES.items()}}

work["product"] = work["product"].str.strip().str.lower().map(product_map)
work["state"] = work["state"].str.strip().str.lower().map(state_map)

In [ ]:
assert work["product"].isna().sum() == 0, "unmapped product spellings → NaN: extend product_map"
assert work["state"].isna().sum() == 0, "unmapped state spellings → NaN: extend state_map"
assert set(work["product"]) == {"Credit Card", "Personal Loan"} and set(work["state"]) == set(STATES)
print("✅ Exercise 16 passed")

### 5b · Employment: case, padding **and corrupt text** (worked example — *fuzzy matching with human review*)
Mapping tables work when you know every variant. For typos like `Fu11-time` we use **fuzzy matching** (`difflib`) — but a detective **reviews the suggested matches by eye** before trusting them.

In [ ]:
CANON = ["Full-time", "Part-time", "Casual", "Self-employed", "Retired", "Unemployed"]
canon_lower = {c.lower(): c for c in CANON}

emp_key = work.employment_status.str.strip().str.lower()
unknown_spellings = sorted(set(emp_key.dropna()) - set(canon_lower))
lookup = {s: (difflib.get_close_matches(s, list(canon_lower), n=1, cutoff=0.6) or [None])[0] for s in unknown_spellings}
display(pd.Series(lookup, name="best fuzzy match"))          # ← REVIEW: do all of these make sense?

In [ ]:
emp_map = {**canon_lower, **{k: canon_lower[v] for k, v in lookup.items() if v}}
work["employment_status"] = emp_key.map(emp_map).fillna("Unknown")           # missing → explicit 'Unknown' category (informative!)

assert set(work.employment_status) <= set(CANON) | {"Unknown"}
print(work.employment_status.value_counts().to_dict())

In [ ]:
assert work.employment_status.eq("Unknown").sum() == 42, "'Unknown' count should equal blank + 'unknown' + 'N/A' = 42"
print("✅ Test passed")

### 5c · Pending from Phase 2 — the `customers` table  *(✍️ Exercise 17)*
Now that `state` has one spelling, split out the second observational unit. **Define:** customer attributes (`customer_id, customer_name, dob, state`) repeat per application. **Fix:** `customers` = those 4 columns, de-duplicated; **Test:** `customer_id` must be **unique** (if not, the same customer has conflicting attributes — investigate!). Then slim `work` down to application-level columns.

In [ ]:
work["customer_id"] = work.customer_id.str.strip()
customers = work[["customer_id", "customer_name", "dob", "state"]].drop_duplicates().reset_index(drop=True)
applications = work.drop(columns=["customer_name", "dob", "state"] + dpd_cols)

In [ ]:
assert customers.customer_id.is_unique, "a customer has conflicting attributes across rows!"
assert len(customers) == 1000, len(customers)
assert not set(dpd_cols) & set(applications.columns)
print("✅ Exercise 17 passed — customers:", len(customers), "| applications:", len(applications), "| performance:", len(performance))

### 5d · Completeness, round 2 — impute *with a documented strategy*
**Define:** income is still `NaN` for the ‘not provided’ rows. **Options:** drop rows · leave NaN · impute. In credit risk, rows are precious and income is informative, so we **impute the median income of applicants with the same employment status & product** and keep `income_missing` so models/analysts know it's not observed. **Not imputed on purpose:** `bureau_score` (leave `NaN`; a missing bureau score is itself a risk signal), `approved_amount` (quarantined), `term_months` (structurally missing for cards), `application_date`.

In [ ]:
grp_median = applications.groupby(["employment_status", "product"])["annual_income"].transform("median")
applications["annual_income"] = applications.annual_income.fillna(grp_median).fillna(applications.annual_income.median())

assert applications.annual_income.notna().all()
assert applications.income_missing.sum() == int(work.income_missing.sum())    # the flag still records what was imputed
print("income NaN left:", applications.annual_income.isna().sum(), "| imputed rows flagged:", int(applications.income_missing.sum()))

---
# ACT VI · Case closed? — prove it
### VI.1 The verdict: turn the data dictionary into executable tests
> A detective doesn't announce “it's clean”; he **proves** it. Every rule in the dictionary becomes an assertion. *(✍️ Exercise 18 — add the three missing rules.)*

In [ ]:
def verdict(apps, custs, perf):
    rules = {
        "app_id unique & well-formed":      apps.app_id.is_unique and apps.app_id.str.fullmatch(r"APP-\d{4}").all(),
        "customer_id unique in customers":  custs.customer_id.is_unique,
        "every application has a customer": apps.customer_id.isin(custs.customer_id).all(),
        "product in allowed list":          apps["product"].isin(["Credit Card", "Personal Loan"]).all(),
        "state in 8 allowed codes":         custs.state.isin(list(STATES)).all(),
        "bureau_score in 300-900 (or NaN)": apps.bureau_score.dropna().between(300, 900).all(),
        "approved <= requested (or NaN)":   (apps.approved_amount.dropna() <= apps.requested_amount[apps.approved_amount.notna()]).all(),
        "application_date within window (or NaT)": apps.application_date.dropna().between("2023-01-01", "2025-06-30").all(),
        "term only for loans (cards have NaN term)": apps.loc[apps["product"] == "Credit Card", "term_months"].isna().all() and apps.loc[apps["product"] == "Personal Loan", "term_months"].notna().all(),
    }
    res = pd.Series(rules, name="passed")
    display(res.to_frame())
    return bool(res.all())

In [ ]:
assert verdict(applications, customers, performance), "At least one rule failed — find it, fix it, re-run (that's Define → Code → Test!)"
print("✅ Exercise 18 passed — all dictionary rules hold")

### VI.2 The reveal — does the mess actually change the answer?
Compare what a **naive analyst** would report from the raw file with the **clean** truth.

In [ ]:
naive = df_naive.copy()                                         # first naive read
naive_income = pd.to_numeric(naive.annual_income, errors="coerce")
naive_default = pd.to_numeric(naive.default_flag, errors="coerce")
naive_score = pd.to_numeric(naive.bureau_score, errors="coerce")

reveal = pd.DataFrame({
    "naive (raw)": {
        "applications": len(naive),
        "distinct states": naive.state.nunique(),
        "distinct products": naive["product"].nunique(),
        "mean bureau score": naive_score.mean(),
        "mean income ($)": naive_income[naive_income > 0].mean(),
        "default rate (flags that parse as 0/1)": naive_default.mean(),
    },
    "clean": {
        "applications": len(applications),
        "distinct states": customers.state.nunique(),
        "distinct products": applications["product"].nunique(),
        "mean bureau score": applications.bureau_score.astype(float).mean(),
        "mean income ($)": applications.annual_income.mean(),
        "default rate (flags that parse as 0/1)": applications.default_flag.mean(),
    },
}).round(3)
display(reveal)

**Your turn to reason like a detective (write 3 sentences):** which of those differences would have changed a *business decision* (e.g. a risk-appetite or pricing decision)? Which error was the most dangerous, and why — the loud one (e.g. 30+ states) or a quiet one (e.g. `9999` inflating the mean score)?

_..._

In [ ]:
# Where does state-level default rate go wrong? Naive grouping fragments each state into several spellings.
print("Naive: default rate by raw `state` spelling (first 8 of", naive.state.nunique(), "groups)")
display(naive.assign(d=naive_default).groupby("state").d.agg(["size", "mean"]).sort_values("size", ascending=False).head(8).round(3))
print("Clean: default rate by state")
display(applications.merge(customers[["customer_id", "state"]], on="customer_id").groupby("state").default_flag.agg(["size", "mean"]).sort_values("size", ascending=False).round(3))

### VI.3 Hand over the evidence: save clean tables + a cleaning log
Good cleaning is **reproducible**. Save the outputs and the audit trail.

In [ ]:
OUT = Path("clean_output"); OUT.mkdir(exist_ok=True)
applications.to_parquet(OUT / "applications_clean.parquet", index=False)
customers.to_parquet(OUT / "customers_clean.parquet", index=False)
performance.to_parquet(OUT / "performance_clean.parquet", index=False)
review_queue.to_csv(OUT / "review_queue_approved_gt_requested.csv", index=False)

cleaning_log = pd.DataFrame(board).assign(status="fixed")
cleaning_log.to_csv(OUT / "cleaning_log.csv", index=False)
print("saved:", sorted(p.name for p in OUT.iterdir()))

---
# Consolidation
### The one-page casebook
| Phase | Question | Typical tool | Trap |
|---|---|---|---|
| Read | what's *really* in the file? | `read_csv(dtype=str, keep_default_na=False)` | pandas pre-interpreting evidence |
| Duplicates | repeated entities? | `duplicated()` on a **normalised key** | near-duplicates |
| Complete | what's missing (or disguised)? | token list, sentinel hunt, `crosstab` of missing-flag | structural missingness; `0/-1/9999` |
| Tidy | 3 rules? | `str.split`, `melt`, split tables | splitting too early/late (dependencies) |
| Valid | legal type/range/format? | `to_numeric/to_datetime(errors="coerce")`, `between` | validity fixes create new NaNs |
| Accurate | true? | cross-field rules, derive from definitions | “can't know → quarantine”, don't fabricate |
| Consistent | one spelling per concept? | mapping dicts, fuzzy match **+ human review** | silent unmapped → NaN |
| Verify | proof? | `assert`s from the data dictionary; before/after reveal | “it looks fine” |

### The 8 habits of a good detective-cleaner
1. Keep the evidence (`raw`); always `.copy()`. 2. Define → Code → Test. 3. Add **flag columns** for anything you change. 4. Document assumptions. 5. Never impute without a stated strategy. 6. Quarantine what you can't resolve. 7. Re-assess after each phase (it's iterative). 8. Prove it with tests, not eyeballs.

### 🧠 Spaced recall — do again tomorrow, in 3 days, in a week (answer *aloud* first)
1. Dirty vs messy — three examples of each.  2. The 3 rules of tidy data.  3. Teacher's cleaning order, and one situation where you'd deviate (and why).  4. Valid vs accurate.  5. What is a sentinel value?  6. Why `keep_default_na=False` during assessment?  7. When do you quarantine instead of fix?  8. Why keep flag columns after cleaning?  9. Why can fixing validity *reduce* completeness?  10. What does Define→Code→Test protect you from?

<details><summary>Answers</summary>

1. Dirty: duplicates, missing, corrupt/inaccurate values. Messy: two variables in one column, values in headers, two entities in one table.
2. Variable→column, observation→row, observational unit→table.
3. Completeness → Tidiness → Validity → Accuracy → Consistency. Deviate: normalise IDs before de-duplicating; standardise categories before imputing by group or splitting out a customer table.
4. Valid obeys the rules; accurate is true (needs a second source).
5. A placeholder value (9999, -1, 0) that stands for “unknown”.
6. So “blank” vs “N/A” vs “unknown” stay visible; types aren't guessed.
7. When you can't determine which value is right (approved > requested).
8. So analysts/models can audit what changed and treat imputed/corrected rows appropriately.
9. Setting out-of-range values to NaN creates missing values.
10. Silent bugs — fixes that look right but quietly change/lose rows.
</details>

### 🚀 Next cases (stretch)
* Re-run the whole pipeline as a single function `clean_applications(raw) -> (applications, customers, performance, review_queue)` and test it on a *freshly* corrupted file.
* Add **outlier detection** to Accuracy (IQR rule or z-score on income) and decide: flag, cap, or leave?
* Compute how much each cleaning phase changed the portfolio default rate (waterfall) — which fix mattered most?